In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

import joblib
import os


# Settings
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

> “Based on statistical association (Cramér’s V), variance explanation (η²), and business relevance, the final churn prediction model faetures

✅ Business assumptions (GLOBAL, DOCUMENTED)

Expected_Total_Lifetime = 36 months
Gross_Margin = 0.6
Retention_Alpha (α) = 0.5
Reacquisition_Cost_Factor = 0.25

### FINAL features for churn prediction

#### Core customer profile
- tenure
- Contract
- SeniorCitizen
- Partner
- Dependents
#### Billing & payment
- MonthlyCharges
- PaymentMethod
- PaperlessBilling
#### Services
- InternetService
- StreamingTV
- StreamingMovies
#### Support / protection
- TechSupport
- OnlineSecurity
- OnlineBackup
- DeviceProtection

In [2]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

#Remove customerID as it's not useful for modeling

df =  df.drop("customerID", axis = 1)

print(f"Dataset Shape: {df.shape}")
print(f"Number of Customers: {df.shape[0]:,}")
print(f"Number of Features: {df.shape[1]}")

Dataset Shape: (7043, 20)
Number of Customers: 7,043
Number of Features: 20


In [3]:
# Convert TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.describe()
#don't mind about senior citizen description

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7043.000000,7032.000000
mean,0.162147,32.371149,64.761692,2283.300441
std,0.368612,24.559481,30.090047,2266.771362
min,0.000000,0.000000,18.250000,18.800000
25%,0.000000,9.000000,35.500000,401.450000
50%,0.000000,29.000000,70.350000,1397.475000
75%,0.000000,55.000000,89.850000,3794.737500
max,1.000000,72.000000,118.750000,8684.800000


In [4]:
print("Missing Values:")
print(df.isnull().sum())
print(f"\nTotal Missing Values: {df.isnull().sum().sum()}")

Missing Values:
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

Total Missing Values: 11


In [5]:
df_clean = df.copy()
# Handle null values
if df_clean['TotalCharges'].isnull().sum() > 0:
    # Fill with median or drop rows
    df_clean = df_clean.dropna(subset=['TotalCharges'])
    print(f"Rows after cleaning: {df_clean.shape[0]}")

Rows after cleaning: 7032


In [6]:
df_clean.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [7]:
numeric_features = [
    "tenure",
    "MonthlyCharges"
]

categorical_features = [
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "Contract",
    "PaymentMethod",
    "PaperlessBilling",
    "InternetService",
    "TechSupport",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "StreamingTV",
    "StreamingMovies"
]

In [8]:
features_to_use = numeric_features + categorical_features
X = df_clean[features_to_use]
X

,tenure,MonthlyCharges,SeniorCitizen,Partner,Dependents,Contract,PaymentMethod,PaperlessBilling,InternetService,TechSupport,OnlineSecurity,OnlineBackup,DeviceProtection,StreamingTV,StreamingMovies
0,1,29.85,0,Yes,No,Month-to-month,Electronic check,Yes,DSL,No,No,Yes,No,No,No
1,34,56.95,0,No,No,One year,Mailed check,No,DSL,No,Yes,No,Yes,No,No
2,2,53.85,0,No,No,Month-to-month,Mailed check,Yes,DSL,No,Yes,Yes,No,No,No
3,45,42.30,0,No,No,One year,Bank transfer (automatic),No,DSL,Yes,Yes,No,Yes,No,No
4,2,70.70,0,No,No,Month-to-month,Electronic check,Yes,Fiber optic,No,No,No,No,No,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,24,84.80,0,Yes,Yes,One year,Mailed check,Yes,DSL,Yes,Yes,No,Yes,Yes,Yes
7039,72,103.20,0,Yes,Yes,One year,Credit card (automatic),Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes
7040,11,29.60,0,Yes,Yes,Month-to-month,Electronic check,Yes,DSL,No,Yes,No,No,No,No
7041,4,74.40,1,Yes,No,Month-to-month,Mailed check,Yes,Fiber optic,No,No,No,No,No,No


In [9]:
#Define Target

y = df_clean["Churn"].map({"Yes": 1, "No": 0})

# Drop target from features 
X = df_clean.drop("Churn", axis=1)

In [10]:
# Train / Validation / Test Split 
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

In [11]:
print("Train:\n", y_train.value_counts())
print("Val:\n", y_val.value_counts())
print("Test:\n", y_test.value_counts())


Train:
 Churn
0    3614
1    1308
Name: count, dtype: int64
Val:
 Churn
0    774
1    281
Name: count, dtype: int64
Test:
 Churn
0    775
1    280
Name: count, dtype: int64


In [12]:
# Build Preprocessing Pipeline 

# Numeric transformer 
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical transformer
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [13]:
# ColumnTransformer 
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [14]:
preprocessor.fit(X_train)

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


In [15]:
X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)


In [16]:
feature_names = preprocessor.get_feature_names_out()
X_train_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)
X_train_df

,num__tenure,num__MonthlyCharges,cat__SeniorCitizen_0,cat__SeniorCitizen_1,cat__Partner_No,cat__Partner_Yes,cat__Dependents_No,cat__Dependents_Yes,cat__Contract_Month-to-month,cat__Contract_One year,...,cat__OnlineBackup_Yes,cat__DeviceProtection_No,cat__DeviceProtection_No internet service,cat__DeviceProtection_Yes,cat__StreamingTV_No,cat__StreamingTV_No internet service,cat__StreamingTV_Yes,cat__StreamingMovies_No,cat__StreamingMovies_No internet service,cat__StreamingMovies_Yes
4499,-0.833469,0.444749,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1933,-0.508058,-1.492135,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4668,-1.240233,-0.120451,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
5681,0.061411,-0.021293,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0
3610,-0.833469,1.166949,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5161,-0.386029,-0.353472,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
3451,1.322379,0.201812,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4135,0.142764,0.927318,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
4249,-0.914822,0.034897,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0


In [17]:
os.makedirs("artifacts", exist_ok=True)

In [18]:
joblib.dump(X_train_processed, "artifacts/X_train.pkl")
joblib.dump(X_val_processed, "artifacts/X_val.pkl")
joblib.dump(X_test_processed, "artifacts/X_test.pkl")

joblib.dump(y_train, "artifacts/y_train.pkl")
joblib.dump(y_val, "artifacts/y_val.pkl")
joblib.dump(y_test, "artifacts/y_test.pkl")

joblib.dump(preprocessor, "artifacts/preprocessor.pkl")

['artifacts/preprocessor.pkl']